# Financial SMS Model 1 — Algorithm Comparison

Compare multiple ML classifiers for **Transaction vs Non-Transaction** using the same TF-IDF features and the same stratified train/test split.

Algorithms:
- Logistic Regression
- Linear SVM
- Multinomial Naive Bayes
- Complement Naive Bayes
- SGD Classifier
- Ridge Classifier
- K-Nearest Neighbors

The notebook compares Accuracy, Precision, Recall, F1, training time, confusion matrices, and unseen SMS predictions.


In [ ]:
# STEP 1 — Setup
!pip install -q pandas numpy scikit-learn openpyxl matplotlib joblib

import os, re, time, unicodedata, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Setup complete.")


In [ ]:
# STEP 2 — Load dataset
DATA_PATH = "bank_sms_rule_based_model.xlsx"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Upload bank_sms_rule_based_model.xlsx")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{DATA_PATH} not found.")

xls = pd.ExcelFile(DATA_PATH)
print("Available sheets:", xls.sheet_names)

df = pd.read_excel(DATA_PATH, sheet_name="Model Predictions")
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))


In [ ]:
# STEP 3 — Clean and create Model 1 labels
if "SMS" not in df.columns or "Type" not in df.columns:
    raise ValueError("Dataset must contain SMS and Type columns.")

df["SMS"] = df["SMS"].fillna("").astype(str).str.strip()
df["Type"] = df["Type"].fillna("").astype(str).str.strip()
df = df[df["SMS"].str.len() > 0].copy()

TRANSACTION_TYPES = {
    "Debit", "Credit", "Transfer (Debit)", "Transfer (Credit)",
    "Deposit", "Withdrawal"
}

def map_ground_truth(value):
    value = str(value).strip()
    if value in TRANSACTION_TYPES:
        return "Transaction"
    if value.startswith("Non-Transaction:"):
        return "Non-Transaction"
    if value == "E-Ticket":
        return "Non-Transaction"
    return "AMBIGUOUS"

df["Ground_Truth_Flag_raw"] = df["Type"].apply(map_ground_truth)

print(df["Ground_Truth_Flag_raw"].value_counts(dropna=False))

ambiguous = df[df["Ground_Truth_Flag_raw"] == "AMBIGUOUS"]
if len(ambiguous):
    print("\nAmbiguous Type values:")
    print(ambiguous["Type"].value_counts())


In [ ]:
# STEP 4 — Build clean ML dataset and remove duplicate/conflicting SMS
model_df = df[
    df["Ground_Truth_Flag_raw"].isin(["Transaction", "Non-Transaction"])
][["SMS", "Ground_Truth_Flag_raw"]].rename(
    columns={"Ground_Truth_Flag_raw": "label"}
).copy()

def normalize_sms(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

model_df["normalized_sms"] = model_df["SMS"].apply(normalize_sms)

conflicts = model_df.groupby("normalized_sms")["label"].nunique()
conflicts = conflicts[conflicts > 1]

print("Conflicting SMS:", len(conflicts))

model_df = model_df[
    ~model_df["normalized_sms"].isin(conflicts.index)
].drop_duplicates("normalized_sms", keep="first").copy()

model_df["text"] = model_df["SMS"].apply(normalize_sms)
model_df = model_df[["SMS", "text", "label"]].reset_index(drop=True)

print("\nFinal ML dataset:", model_df.shape)
print(model_df["label"].value_counts())


In [ ]:
# STEP 5 — Same train/test split for every algorithm
X = model_df["text"]
y = model_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training:", len(X_train))
print("Testing :", len(X_test))


In [ ]:
# STEP 6 — Define common TF-IDF and all algorithms
def make_tfidf():
    return TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
        max_features=20000
    )

models = {
    "Logistic Regression": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Linear SVM": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", LinearSVC(
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Multinomial Naive Bayes": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", MultinomialNB())
    ]),

    "Complement Naive Bayes": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", ComplementNB())
    ]),

    "SGD Classifier": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", SGDClassifier(
            loss="hinge",
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Ridge Classifier": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", RidgeClassifier(
            class_weight="balanced"
        ))
    ]),

    "K-Nearest Neighbors": Pipeline([
        ("tfidf", make_tfidf()),
        ("classifier", KNeighborsClassifier(n_neighbors=5))
    ])
}

print("Algorithms:", list(models.keys()))


In [ ]:
# STEP 7 — Train and evaluate every algorithm
comparison_results = []
trained_models = {}
prediction_store = {}

for name, model in models.items():
    print("\n" + "="*70)
    print("TRAINING:", name)

    start = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start

    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, pos_label="Transaction", zero_division=0)
    recall = recall_score(y_test, pred, pos_label="Transaction", zero_division=0)
    f1 = f1_score(y_test, pred, pos_label="Transaction", zero_division=0)

    comparison_results.append({
        "Model": name,
        "Accuracy": acc * 100,
        "Precision": precision * 100,
        "Recall": recall * 100,
        "F1 Score": f1 * 100,
        "Training Time (sec)": training_time
    })

    trained_models[name] = model
    prediction_store[name] = pred

    print(f"Accuracy : {acc*100:.2f}%")
    print(f"Precision: {precision*100:.2f}%")
    print(f"Recall   : {recall*100:.2f}%")
    print(f"F1 Score : {f1*100:.2f}%")
    print(f"Training : {training_time:.4f} sec")


In [ ]:
# STEP 8 — Final comparison table
comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.sort_values(
    "F1 Score", ascending=False
).reset_index(drop=True)

print("="*90)
print("MODEL 1 — ALGORITHM COMPARISON")
print("="*90)

display(comparison_df.style.format({
    "Accuracy": "{:.2f}%",
    "Precision": "{:.2f}%",
    "Recall": "{:.2f}%",
    "F1 Score": "{:.2f}%",
    "Training Time (sec)": "{:.4f}"
}))

best_model_name = comparison_df.iloc[0]["Model"]
print("\nBest by F1 Score:", best_model_name)


In [ ]:
# STEP 9 — Rank by transaction recall
recall_df = comparison_df.sort_values(
    "Recall", ascending=False
).reset_index(drop=True)

display(recall_df[
    ["Model", "Recall", "Precision", "F1 Score", "Accuracy"]
].style.format({
    "Recall": "{:.2f}%",
    "Precision": "{:.2f}%",
    "F1 Score": "{:.2f}%",
    "Accuracy": "{:.2f}%"
}))


In [ ]:
# STEP 10 — Confusion matrices
labels = ["Non-Transaction", "Transaction"]

for name, model in trained_models.items():
    pred = prediction_store[name]
    cm = confusion_matrix(y_test, pred, labels=labels)

    print("\n" + name)
    print(cm)

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels
    ).plot()
    plt.title(name)
    plt.tight_layout()
    plt.show()


In [ ]:
# STEP 11 — Detailed report for the best model
best_model = trained_models[best_model_name]
best_pred = prediction_store[best_model_name]

print("="*70)
print("BEST MODEL:", best_model_name)
print("="*70)

print(classification_report(
    y_test,
    best_pred,
    zero_division=0
))

misclassified = pd.DataFrame({
    "SMS": X_test.values,
    "Actual": y_test.values,
    "Predicted": best_pred
})

misclassified = misclassified[
    misclassified["Actual"] != misclassified["Predicted"]
].reset_index(drop=True)

print("Misclassified:", len(misclassified))

for i, row in misclassified.iterrows():
    print("\n", i+1)
    print("Actual   :", row["Actual"])
    print("Predicted:", row["Predicted"])
    print("SMS      :", row["SMS"])


In [ ]:
# STEP 12 — Test unseen SMS with every algorithm
new_sms = [
    "Dear Sir/Madam, Your A/C 065-2001****11 has been debited by Rs. 1500.00 (ATM @10:30 28/08/2026)",
    "LKR 5000.00 credited to Ac No:01602XXXXX99 on 28/08/26 Reason:CEFT-APPA",
    "POS/ATM Transaction Rs 1030.00 From A/C No XXXXXXXXXX875. Balance available Rs 939.21",
    "Congratulations! You have won exciting reward points. Claim your special offer now.",
    "Get unlimited internet data today. Special promotional offer available now.",
    "Dear customer, please do not share your OTP or account details with anyone."
]

for i, sms in enumerate(new_sms, 1):
    print("\n" + "-"*70)
    print("SMS", i, ":", sms)

    for name, model in trained_models.items():
        pred = model.predict([normalize_sms(sms)])[0]
        print(f"{name:28s} -> {pred}")


In [ ]:
# STEP 13 — Retrain the selected algorithm on ALL clean data
final_model = models[best_model_name]

final_model.fit(
    model_df["text"],
    model_df["label"]
)

print("Final selected algorithm:", best_model_name)
print("Final training samples:", len(model_df))


In [ ]:
# STEP 14 — Save comparison, dataset and final model
joblib.dump(
    final_model,
    "financial_sms_model1_final.joblib"
)

comparison_df.to_csv(
    "model1_algorithm_comparison.csv",
    index=False
)

model_df.to_csv(
    "model1_clean_training_data.csv",
    index=False
)

print("Saved:")
print("- financial_sms_model1_final.joblib")
print("- model1_algorithm_comparison.csv")
print("- model1_clean_training_data.csv")


In [ ]:
# STEP 15 — Export Android parameters when supported
# Linear models can be exported in the same TF-IDF + coefficients structure.
tfidf = final_model.named_steps["tfidf"]
classifier = final_model.named_steps["classifier"]

if hasattr(classifier, "coef_"):
    android_model = {
        "model_type": best_model_name,
        "classes": classifier.classes_.tolist(),
        "vocabulary": tfidf.vocabulary_,
        "idf": tfidf.idf_.tolist(),
        "coefficients": classifier.coef_.tolist(),
        "intercept": classifier.intercept_.tolist(),
        "ngram_range": list(tfidf.ngram_range),
        "lowercase": tfidf.lowercase
    }

    with open("model1_android.json", "w", encoding="utf-8") as f:
        json.dump(android_model, f)

    print("Created: model1_android.json")
    print("Vocabulary size:", len(tfidf.vocabulary_))
else:
    print(
        "The selected algorithm does not expose linear coefficients. "
        "Use its .joblib model for desktop/Python deployment or "
        "perform a separate Android conversion."
    )


## Decision rule

Do not select a model from accuracy alone.

Use:
1. **F1 Score** as the primary overall metric.
2. **Transaction Recall** because missing a real transaction is important.
3. Precision to control false transaction detections.
4. Accuracy as a supporting metric.
5. Android deployment practicality.

After running this notebook, the comparison table provides the evidence for the final Model 1 algorithm.
